In [ ]:
! pip install pandas
! pip install pymongo
! pip install dotenv

In [ ]:
! pip install pymongo
! pip install openai



In [ ]:
from openai import OpenAI
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

from pymongo import MongoClient
client = MongoClient("192.168.2.70", 27017)

mydb = client["messages"]
mycol = mydb["emailmessages3"]

In [ ]:
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))
print("OpenAI API Key:", os.getenv("OPENAI_API_KEY"))


def makeSMS( prompt):
    completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": prompt
        }
    ]
    )
    #print(type(completion.choices[0].messages))
    return(completion.choices[0].message.content)
travelPrompt = "Generate a realistic email message that includes travel-related information please include sender and reciever with different made-up fake names."
financialPrompt = "Generate a realistic email message that includes a financial transaction please include sender and reciever with different made-up fake names."
otherPrompt = "Generate a realistic email message for general notifications or offers please include sender and reciever with different made-up fake names."

totalRecords = 1000
travelCount = 0
finanicalCount = 0
otherCount = 0
promptFlag = 0
message = {}
messageTotal = pd.DataFrame()


for record in range (totalRecords):
    if(record % 10 == 0):
        msg = travelPrompt
        travelCount += 1
        promptFlag = 1
    elif(record % 12 == 0):
        msg =financialPrompt
        finanicalCount += 1
        promptFlag = 2
    else:
        msg=otherPrompt
        promptFlag = 0
        otherCount += 1
        
    msgReply = makeSMS(msg)
    message ={"Message":msgReply, "Type":promptFlag}   
    df_dictionary = pd.DataFrame([message])
    messageTotal = pd.concat([messageTotal, df_dictionary], ignore_index=True)
    
    myentry = { "message": msgReply, "type": promptFlag }
    x = mycol.insert_one(myentry)

    #print("Others: ", otherCount, " Financial:", finanicalCount, " Travel:", travelCount)
       
print(len(messageTotal))
messageTotal.to_csv